In [1]:
import numpy as np
from LanzaModels import TVL1_1D
from ADMMsRustici import SpectralSolver
from signalClass import *
import time

In [2]:
np.random.seed(24102001)
n = 1024

construct blur matrix

In [3]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [4]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 100)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL1 model

In [5]:
mu = 1
VarModel = TVL1_1D.TVL1_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [6]:
#begin solver construction

xk = np.copy(xCorrupted)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

MySolver = SpectralSolver.SpectralSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [7]:
iters = 200

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [8]:
for iter in range(0, iters):

    print(f"{iter} / {iters}")

    sTime = time.process_time_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.process_time_ns()

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ VarModel.Q @ (yk - yk_1))

    XsolutionHistory[iter, :] = xk
    YsolutionHistory[iter, :] = yk
    lambdaHistory[iter, :] = lk
    betaHistory[iter] = betak

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    CpuTimes[iter] = ( (eTime - sTime) / 1e9 ) + CpuTimes[iter - 1]

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1


0 / 200
1 / 200
2 / 200
3 / 200
4 / 200
5 / 200
6 / 200
7 / 200
8 / 200
9 / 200
10 / 200
11 / 200
12 / 200
13 / 200
14 / 200
15 / 200
16 / 200
17 / 200
18 / 200
19 / 200
20 / 200
21 / 200
22 / 200
23 / 200
24 / 200
25 / 200
26 / 200
27 / 200
28 / 200
29 / 200
30 / 200
31 / 200
32 / 200
33 / 200
34 / 200
35 / 200
36 / 200
37 / 200
38 / 200
39 / 200
40 / 200
41 / 200
42 / 200
43 / 200
44 / 200
45 / 200
46 / 200
47 / 200
48 / 200
49 / 200
50 / 200
51 / 200
52 / 200
53 / 200
54 / 200
55 / 200
56 / 200
57 / 200
58 / 200
59 / 200
60 / 200
61 / 200
62 / 200
63 / 200
64 / 200
65 / 200
66 / 200
67 / 200
68 / 200
69 / 200
70 / 200
71 / 200
72 / 200
73 / 200
74 / 200
75 / 200
76 / 200
77 / 200
78 / 200
79 / 200
80 / 200
81 / 200
82 / 200
83 / 200
84 / 200
85 / 200
86 / 200
87 / 200
88 / 200
89 / 200
90 / 200
91 / 200
92 / 200
93 / 200
94 / 200
95 / 200
96 / 200
97 / 200
98 / 200
99 / 200
100 / 200
101 / 200
102 / 200
103 / 200
104 / 200
105 / 200
106 / 200
107 / 200
108 / 200
109 / 200
110 / 200


In [9]:

xReconstr = XsolutionHistory[iters - 1, :]
ConvergenceDistance = np.zeros(shape=(iters,))

ConvergenceDistance = np.linalg.norm(XsolutionHistory - xReconstr, axis=1)

In [11]:
np.savez_compressed(
    "./SpectralADMMTVL1-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,

    ConvergenceDistance = ConvergenceDistance,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
)